# 👤 NovaPay · Base de Datos de Clientes
### Construcción de perfiles de comportamiento para el panel del analista

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("C:/Users/Usuario/Desktop/the bridge/github/Desafio_Grupo1/data/synthetic_fin_data_CLEAN.csv")

In [3]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,hour_of_the_day,ip_country,merchant_category,balance_discrepancy
0,162,CASH_OUT,183806.32,C691771226,19391.00,0.00,C1416312719,382572.19,566378.51,0,18,DE,transport,NaN
1,137,PAYMENT,521.37,C203378011,0.00,0.00,M42773300,0.00,0.00,0,17,ES,transport,NaN
2,179,PAYMENT,3478.18,C1698571270,19853.00,16374.82,M643984524,0.00,0.00,0,11,DE,fuel,NaN
3,355,PAYMENT,1716.05,C913764937,5769.17,4053.13,M1387429131,0.00,0.00,0,19,FR,pharmacy,NaN
4,354,CASH_IN,253129.93,C2017736577,1328499.49,1581629.42,C407484102,2713220.48,2460090.55,0,18,FR,grocery,NaN


In [4]:
df.shape

(317223, 14)

In [5]:
customer_db = df.groupby("nameOrig").agg(
    total_transactions  = ("amount", "count"),
    total_volume        = ("amount", "sum"),
    avg_amount          = ("amount", "mean"),
    max_amount          = ("amount", "max"),
    first_seen          = ("step", "min"),
    last_seen           = ("step", "max"),
    fraud_rate_historical = ("isFraud", "mean"),
    distinct_counterparties = ("nameDest", "nunique"),
    most_used_type      = ("type", lambda x: x.value_counts().index[0]),
).reset_index()

customer_db = customer_db.rename(columns={"nameOrig": "client_id"})

print(customer_db.shape)
print(customer_db.head())

(273195, 10)
     client_id  total_transactions  total_volume  avg_amount  max_amount  \
0  C1000013879                   1     516459.01   516459.01   516459.01   
1   C100001401                   1      17686.93    17686.93    17686.93   
2  C1000018432                   1      12216.58    12216.58    12216.58   
3  C1000036340                   1     253648.68   253648.68   253648.68   
4  C1000040770                   1      15486.87    15486.87    15486.87   

   first_seen  last_seen  fraud_rate_historical  distinct_counterparties  \
0         192        192                    0.0                        1   
1         302        302                    0.0                        1   
2         138        138                    0.0                        1   
3         655        655                    1.0                        1   
4         185        185                    0.0                        1   

  most_used_type  
0       CASH_OUT  
1        PAYMENT  
2        PAYMENT

In [6]:
from datetime import datetime, timedelta

base_date = datetime(2024, 1, 1)

customer_db["first_seen"] = customer_db["first_seen"].apply(
    lambda x: base_date + timedelta(hours=x)
)

customer_db["last_seen"] = customer_db["last_seen"].apply(
    lambda x: base_date + timedelta(hours=x)
)

print(customer_db[["client_id", "first_seen", "last_seen"]].head())

     client_id          first_seen           last_seen
0  C1000013879 2024-01-09 00:00:00 2024-01-09 00:00:00
1   C100001401 2024-01-13 14:00:00 2024-01-13 14:00:00
2  C1000018432 2024-01-06 18:00:00 2024-01-06 18:00:00
3  C1000036340 2024-01-28 07:00:00 2024-01-28 07:00:00
4  C1000040770 2024-01-08 17:00:00 2024-01-08 17:00:00


In [7]:
print(customer_db.columns.tolist())
print(customer_db.dtypes)

['client_id', 'total_transactions', 'total_volume', 'avg_amount', 'max_amount', 'first_seen', 'last_seen', 'fraud_rate_historical', 'distinct_counterparties', 'most_used_type']
client_id                          object
total_transactions                  int64
total_volume                      float64
avg_amount                        float64
max_amount                        float64
first_seen                 datetime64[ns]
last_seen                  datetime64[ns]
fraud_rate_historical             float64
distinct_counterparties             int64
most_used_type                     object
dtype: object


In [8]:
# JSON

customer_db.to_json("C:/Users/Usuario/Desktop/the bridge/github/Desafio_Grupo1/data/customer_db.json", orient="records", date_format="iso")


print("exported successfully")

exported successfully
